In [1]:
!pip install -q opencv-python-headless torch torchvision matplotlib xlsxwriter
from google.colab import drive

# Monta o drive (force_remount=True garante que limpa o cache ao montar)
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import glob
import shutil
from pathlib import Path

# Raiz do projeto no Drive. Definida UMA vez aqui e reaproveitada nas celulas seguintes.
PROJECT_ROOT = Path(r"/content/drive/Shareddrives/Departamentos Externos/Financeiro & Jurídico/CORRETOR DE SIMULADOS")

# Os .tif ficam sempre em data/entrada - nao precisa editar caminho para trocar de prova.
origem  = PROJECT_ROOT / "data" / "entrada"
destino = "/content/dados_locais/originais"

# Garante que data/entrada exista. Num clone novo ela ainda nao existe (data/ e
# ignorada pelo git), e esta celula roda antes de qualquer import do pacote.
os.makedirs(origem, exist_ok=True)

# Esvazia a pasta local antes de copiar para nao misturar simulados
if os.path.exists(destino):
    shutil.rmtree(destino)
os.makedirs(destino, exist_ok=True)

arquivos = glob.glob(os.path.join(str(origem), "*.tif*"))
for arq in arquivos:
    shutil.copy2(arq, destino)

qtd = len(os.listdir(destino))
print(f"Origem: {origem}")
print(f"Ficheiros copiados: {qtd}")
if qtd == 0:
    print()
    print("Nenhum .tif encontrado. Suba os cartoes escaneados em:")
    print(f"   {origem}")
    print("(a pasta acabou de ser criada, se ainda nao existia)")
else:
    print("AVISO: Se o numero for diferente do esperado e tem a certeza do upload, e erro de")
    print("sincronizacao do cache do Drive. Aguarde alguns minutos e rode esta celula de novo.")

In [ ]:
import os
import sys

# Trabalha a partir da raiz do projeto e poe ela no path para que "import corretor" funcione.
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Diretorio de trabalho configurado:', PROJECT_ROOT)

In [ ]:
import cv2
import glob
import os
from pathlib import Path
from corretor.config import CONFIG_SIMULADOS, SIMULADO_ATIVO
from corretor.visao.alinhar_gabarito import alinhar_gabarito

def gerar_teste_visual_grade():
    print(f"Executando teste visual para o simulado ativo: {SIMULADO_ATIVO}")

    # Pega os parâmetros do simulado ativo (SEMI ou CASDINHO)
    config_atual = CONFIG_SIMULADOS.get(SIMULADO_ATIVO, CONFIG_SIMULADOS["CASDINHO"])
    grade_cfg = config_atual["GRADE_RESPOSTAS"]

    # Caminho onde estão os TIFFs locais
    pasta_originais = Path("/content/dados_locais/originais")
    arquivos = sorted(glob.glob(str(pasta_originais / "*.tif*")))

    if not arquivos:
        print("ERRO: Nenhum arquivo encontrado em /content/dados_locais/originais. Execute a cópia primeiro.")
        return

    # Pega o primeiro arquivo para teste
    primeiro_arquivo = arquivos[0]
    print(f"Processando imagem de teste: {os.path.basename(primeiro_arquivo)}")

    # 1. Alinha a imagem
    img_alinhada = alinhar_gabarito(primeiro_arquivo)
    if img_alinhada is None:
        print("ERRO: Falha no alinhamento da imagem de teste (cantos não encontrados).")
        return

    # Converte para BGR para podermos desenhar linhas coloridas por cima
    img_visual = cv2.cvtColor(img_alinhada, cv2.COLOR_GRAY2BGR)

    # 2. Desenha a grade com base nas configurações de blocos
    y_ini = grade_cfg["y_questoes_ini"]
    passo_y = grade_cfg["passo_y_divisor"]

    questao_atual = 1
    for bloco_idx, bloco in enumerate(grade_cfg["blocos"]):
        x_min, x_max = bloco["x"]
        linhas = bloco["linhas"]

        for linha in range(linhas):
            if questao_atual > config_atual["NUM_QUESTOES"]:
                break

            y_centro = y_ini + (linha * passo_y)

            # Desenha uma linha horizontal indicativa da questão
            cv2.line(img_visual, (x_min - 10, y_centro), (x_max + 10, y_centro), (0, 255, 0), 1)

            # Escreve o número da questão ao lado do bloco
            cv2.putText(img_visual, f"Q{questao_atual}", (x_min - 45, y_centro + 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 255), 1)

            # Simula as 5 alternativas (A, B, C, D, E) espaçadas horizontalmente na linha
            largura_coluna = (x_max - x_min) / 5
            for alt_idx in range(5):
                x_alt = int(x_min + (alt_idx * largura_coluna) + (largura_coluna / 2))
                # Desenha um círculo pequeno onde o script mede a intensidade da bolinha
                cv2.circle(img_visual, (x_alt, y_centro), 3, (255, 0, 0), -1)

            questao_atual += 1

    # 3. Salva a imagem de teste gerada para inspeção visual
    pasta_saida = Path("/content/dados_locais/teste_visual")
    pasta_saida.mkdir(parents=True, exist_ok=True)

    caminho_saida = pasta_saida / f"grid_debug_{SIMULADO_ATIVO}.jpg"
    cv2.imwrite(str(caminho_saida), img_visual)

    print(f"✓ Teste visual gerado com sucesso!")
    print(f"-> Abra o painel lateral esquerdo do Colab e descarregue a imagem em: {caminho_saida}")

if __name__ == "__main__":
    gerar_teste_visual_grade()

In [ ]:
# @title ⚙️ Configuração do Gabarito Oficial
# @markdown Dê nome ao simulado, escolha o modelo e insira as alternativas correspondentes ao gabarito. Se você, bixão, errar as alternativas, corrija aqui e gere a planilha novamente(50 letras para CASDINHO):

nome_simulado = "SEMI 1" # @param {type:"string"}
simulado_selecionado = "SEMI" # @param ["CASDINHO", "SEMI"]
gabarito_oficial = "DBCDBEDCBDDCACACCBDDCCEBCCEEDCBBEBBEACABCDBCCCDCCCADDBCCCDAC" # @param {type:"string"}

import os
os.makedirs('/content/dados_locais', exist_ok=True)

with open('/content/dados_locais/nome_simulado.txt', 'w', encoding='utf-8') as f:
    f.write(nome_simulado.strip())

with open('/content/dados_locais/simulado_ativo.txt', 'w') as f:
    f.write(simulado_selecionado)

gabarito_oficial = gabarito_oficial.upper().replace(" ", "").strip()
with open('/content/dados_locais/gabarito_atual.txt', 'w') as f:
    f.write(gabarito_oficial)

print(f"✓ Sistema configurado para o modelo: {simulado_selecionado}")
print("✓ Gabarito salvo na memória!")

In [ ]:
# @title ✂️ Processar Imagens
from corretor.visao.extracao_em_lote import processar_simulados

print("Iniciando extração e leitura dos gabaritos...")
processar_simulados(extrair_respostas=True, extrair_inscricao=True)

In [ ]:
# @title 📊 Gerar Planilha de Resultados
from corretor.relatorio.gerar_excel import gerar_excel_final

print("Consolidando notas e gerando Excel...")
gerar_excel_final()